# v9 - Division-Aware Tracking - Phase 0 (oracle audit / mandatory gate)

Plan: `docs/v9_division_aware_tracking_plan.md`.

**What this notebook does (Phase 0).** It (optionally) GENERATES baseline predicted graphs
(`.geff`) with the frozen detector+transformer+ILP on a held-out subset, then measures - with the
**official scorer** - the ceiling of division recovery under three candidate families:

- **A** orphan-only: add `M -> D2` where `D2` has no incoming edge.
- **B** steal a weak child (scaffold).
- **C** joint daughter selection (scaffold).

For each family it reports, per specimen (`44b6` / `6bba`): recoverable GT divisions, oracle
`div_J`, `Delta adj`, `Delta score = Delta adj + 0.1 * Delta div_J`. **GATE 0** then decides whether
to build Phase 1.

**How to run.** GPU ON. Attach: (1) competition data (train `<id>.zarr` + `<id>.geff`), (2) the
support-pack dataset (`repo/` + `weights/` + `wheels/`). Internet ON is fine (local audit, not a
submission). Cell 0.2b does the one GPU pass; everything after it is CPU and fast.

**Scope note.** The gate needs only `pred .geff + GT + scorer`. The pre-ILP candidate-edge export
(transformer logits/ranks/margins) is a Phase-1 feature concern (final optional cell). Absolute
local numbers are optimistic (audit runs on the split_0 held-out test, but the extractors still saw
all train videos); trust per-specimen consistency and deltas, not absolutes.

In [1]:
# --- Phase 0.1 : environment ---------------------------------------------------
import importlib, sys, glob, subprocess
from pathlib import Path

def _find(pattern):
    return sorted(glob.glob(pattern, recursive=True))

# Support-pack repo = the dir that contains scripts/predict_unet_transformer.py
_hits = _find("/kaggle/input/**/scripts/predict_unet_transformer.py")
assert _hits, "support-pack repo not found under /kaggle/input (attach the pack dataset)."
REPO = Path(_hits[0]).parent.parent
for p in (REPO / "src", REPO / "scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
print("REPO =", REPO)

# Dependency resolution is deliberately disabled so pip cannot replace Kaggle's
# already-imported numpy/scipy and create a binary ABI mismatch. Because --no-deps
# is used, every runtime dependency must be named explicitly. This list is the
# dependency closure validated by the v7 offline gate and the v8 submission.
PIP_SPECS = [
    "tracksdata", "pyscipopt", "ilpy>=0.5.1",
    "zarr>=3.0.10,<4", "geff>=1.1.3.1.1", "geff-spec<1.2",
    "polars>=1.36", "blosc2", "dask", "imagecodecs", "scikit-image>=0.24",
    "pyarrow", "rustworkx>=0.17.1", "sqlalchemy>=2", "numcodecs>=0.13,<0.16",
    "donfig>=0.8", "google-crc32c>=1.5", "bidict>=0.23.1", "psygnal>=0.14",
    "rich", "networkx>=3.2.1", "pydantic>=2.11", "pydantic-core",
    "annotated-types", "typing-extensions>=4.13", "typing-inspection",
    "markdown-it-py", "pygments", "click", "cloudpickle", "fsspec",
    "partd", "locket", "toolz", "pyyaml", "ndindex", "msgpack",
    "numexpr", "deprecated", "wrapt",
]

CRITICAL_MODULES = ("tracksdata", "geff", "geff_spec", "zarr",
                    "pyscipopt", "ilpy", "donfig", "numcodecs",
                    "polars", "blosc2", "dask", "imagecodecs",
                    "pyarrow", "rustworkx", "sqlalchemy", "skimage")

def _clear_partial_imports():
    # A failed import can leave half-initialized packages in sys.modules.
    roots = set(CRITICAL_MODULES) | {"skimage"}
    for name in list(sys.modules):
        if any(name == root or name.startswith(root + ".") for root in roots):
            sys.modules.pop(name, None)
    importlib.invalidate_caches()

def _import_failures():
    failures = {}
    for name in CRITICAL_MODULES:
        try:
            importlib.import_module(name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"
    try:
        import zarr as _zarr
        if int(_zarr.__version__.split(".")[0]) < 3:
            failures["zarr"] = f"zarr {_zarr.__version__} is too old; need >=3"
    except Exception:
        pass
    try:
        import polars as _pl
        polars_ok = (hasattr(_pl, "Float16") and
                     _pl.Series([-999999.0], dtype=_pl.Float64).dtype == _pl.Float64)
        if not polars_ok:
            failures["polars"] = f"polars {_pl.__version__} is too old"
    except Exception:
        pass
    return failures

def _run_pip(command, label):
    print(label)
    result = subprocess.run(command, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout[-4000:])
    if result.returncode != 0:
        print(result.stderr[-4000:])
    return result.returncode == 0

failures = _import_failures()
if failures:
    print("Dependency check failed:", failures)
    wheel_dirs = []
    for path in [REPO.parent / "wheels", *map(Path, _find("/kaggle/input/**/wheels"))]:
        if path.is_dir() and path not in wheel_dirs:
            wheel_dirs.append(path)

    base = [sys.executable, "-m", "pip", "install", "--no-deps"]
    installed = False
    if wheel_dirs:
        offline = base + ["--no-index"]
        for path in wheel_dirs:
            offline += ["--find-links", str(path)]
        installed = _run_pip(offline + PIP_SPECS,
                             f"Installing from offline wheels: {wheel_dirs}")
    if not installed:
        installed = _run_pip(base + PIP_SPECS, "Offline install unavailable; trying PyPI")
    if not installed:
        raise RuntimeError("Dependency installation failed; see pip output above.")

    _clear_partial_imports()
    failures = _import_failures()
    if failures:
        raise ImportError(f"Dependencies still fail after installation: {failures}")

import tracksdata, geff, zarr  # noqa: F401
print("tracksdata", getattr(tracksdata, "__version__", "?"),
      "| geff", getattr(geff, "__version__", "?"),
      "| zarr", getattr(zarr, "__version__", "?"))


REPO = /kaggle/input/datasets/pilkwang/biohub-temporal-unet3d-seed314159-v1/repo


ModuleNotFoundError: No module named 'donfig'

In [ ]:
# --- Phase 0.2 : imports -------------------------------------------------------
import json
import numpy as np
import polars as pl
import tracksdata as td
from collections import defaultdict, Counter

try:
    from geff import GeffMetadata
except Exception:
    GeffMetadata = None

from biohub_tracking.io import open_dataset, save_graph
from biohub_tracking.metrics import evaluate as compute_metric, per_sample_metrics
from predict_unet_transformer import load_model, predict_video, build_graph, PredictConfig

K = td.DEFAULT_ATTR_KEYS
print("attr keys:", K.NODE_ID, K.T, K.EDGE_SOURCE, K.EDGE_TARGET, K.MATCHED_NODE_ID)

def specimen_of(name):
    return name.split("_")[0]

In [ ]:
# --- Phase 0.2b : GENERATE baseline .geff (GPU, one pass) --------------------
# Runs the frozen detector+transformer+ILP on a held-out subset (split_0 test,
# division-richest) and saves per-video .geff to /kaggle/working/preds.
# Set GENERATE_BASELINE=False if you already have baseline predictions to audit
# (then set PRED_DIR in cell 0.3).
import torch

GENERATE_BASELINE  = True
GEN_DIR            = Path("/kaggle/working/preds"); GEN_DIR.mkdir(parents=True, exist_ok=True)
GEN_N_PER_SPECIMEN = 8          # videos per specimen to generate + audit
DET_THRESHOLD      = 0.96875    # matches the shipped 0.912 detection threshold
PICK_DIVISION_RICH = True       # rank candidates by GT division count (richer oracle signal)

# train data dir (has <id>.zarr + <id>.geff)
_zdirs = sorted({str(Path(p).parent) for p in _find("/kaggle/input/**/*.zarr")})
DATA_DIR = Path(_zdirs[0]) if _zdirs else None
assert DATA_DIR is not None, "train data dir (with <id>.zarr + <id>.geff) not found."
print("DATA_DIR =", DATA_DIR)

def _gt_div_count(name):
    try:
        gt = open_dataset(DATA_DIR / name, require_tracks=True).tracks
        if gt.num_edges() == 0:
            return 0
        df = gt.edge_attrs(attr_keys=[K.EDGE_SOURCE])
        c = Counter(int(s) for s in df[K.EDGE_SOURCE].to_list())
        return sum(1 for v in c.values() if v >= 2)
    except Exception:
        return 0

if GENERATE_BASELINE:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    _w = [w for w in _find("/kaggle/input/**/edge_predictor_best.pth")
          if (Path(w).parent / "config.json").exists()]
    assert _w, "edge_predictor_best.pth (with sibling config.json) not found."
    WEIGHTS = Path(_w[0]); print("WEIGHTS =", WEIGHTS, "| device =", device)
    model, window_size, downsample = load_model(WEIGHTS, device)

    # candidate videos = split_0 test (held out for THIS model) if the splits file exists
    splits_file = DATA_DIR / "dataset_splits.json"
    if splits_file.exists():
        cand = json.loads(splits_file.read_text())[0]["test"]
        print("using split_0 test:", len(cand), "videos")
    else:
        cand = [p.stem for p in sorted(DATA_DIR.glob("*.zarr"))]
        print("no splits file -> using all:", len(cand), "videos")
    cand = [n for n in cand
            if (DATA_DIR / f"{n}.geff").exists() and (DATA_DIR / f"{n}.zarr").exists()]

    by = defaultdict(list)
    if PICK_DIVISION_RICH:
        for dc, n in sorted(((_gt_div_count(n), n) for n in cand), reverse=True):
            by[specimen_of(n)].append(n)
    else:
        for n in cand:
            by[specimen_of(n)].append(n)
    GEN_NAMES = [n for s in sorted(by) for n in by[s][:GEN_N_PER_SPECIMEN]]
    print("generating", len(GEN_NAMES), "videos:",
          {s: sum(specimen_of(n) == s for n in GEN_NAMES) for s in set(map(specimen_of, GEN_NAMES))})

    cfg = PredictConfig(
        det_threshold=DET_THRESHOLD, det_tta=True,
        edge_activation="softmax", threshold=0.5,
        use_ilp=True, ilp_edge_weight=-1.0,
        ilp_appearance_weight=0.1, ilp_disappearance_weight=0.1,
        ilp_division_weight=1.0,
    )
    for name in GEN_NAMES:
        out = GEN_DIR / f"{name}.geff"
        if out.exists():
            print("  skip (exists):", name); continue
        coords, edges = predict_video(model, DATA_DIR / name, device, cfg=cfg,
                                      window_size=window_size, downsample=downsample)
        g = build_graph(coords, edges)
        if cfg.use_ilp and g.num_edges() > 0:
            solver = td.solvers.ILPSolver(
                edge_weight=cfg.ilp_edge_weight * td.EdgeAttr("edge_prob"),
                appearance_weight=cfg.ilp_appearance_weight,
                disappearance_weight=cfg.ilp_disappearance_weight,
                division_weight=cfg.ilp_division_weight,
            )
            g = solver.solve(g)
        save_graph(g, out)
        print("  saved", name, "| nodes", g.num_nodes(), "edges", g.num_edges())
    PRED_DIR = GEN_DIR
    print("PRED_DIR =", PRED_DIR)

In [ ]:
# --- Phase 0.3 : config / video list ----------------------------------------
VOXEL_SCALE = (1.625, 0.40625, 0.40625)   # (z, y, x) um
MAX_DIST    = 7.0                           # node-match gate (um)
DIV_WEIGHT  = 0.1                           # metric division weight
SUBSET_PER_SPECIMEN = None                  # None = use all generated videos

# DATA_DIR / PRED_DIR normally come from cell 0.2b. If you skipped generation,
# set PRED_DIR to your baseline *.geff folder (and DATA_DIR to the train dir).
try:
    DATA_DIR
except NameError:
    _z = sorted({str(Path(p).parent) for p in _find("/kaggle/input/**/*.zarr")})
    DATA_DIR = Path(_z[0]) if _z else None
try:
    PRED_DIR
except NameError:
    PRED_DIR = None
if PRED_DIR is None:
    _p = [d for d in sorted({str(Path(p).parent) for p in _find("/kaggle/input/**/*.geff")})
          if DATA_DIR is None or Path(d) != DATA_DIR]
    PRED_DIR = Path(_p[0]) if _p else None
assert DATA_DIR is not None and PRED_DIR is not None, "set DATA_DIR and PRED_DIR"
print("DATA_DIR =", DATA_DIR); print("PRED_DIR =", PRED_DIR)

names = [p.stem for p in sorted(Path(PRED_DIR).glob("*.geff"))
         if (DATA_DIR / f"{p.stem}.geff").exists()]
if SUBSET_PER_SPECIMEN:
    by = defaultdict(list)
    for n in names:
        by[specimen_of(n)].append(n)
    names = [n for g in by.values() for n in g[:SUBSET_PER_SPECIMEN]]
print(f"{len(names)} videos:",
      {s: sum(specimen_of(n) == s for n in names) for s in set(map(specimen_of, names))})

In [ ]:
# --- Phase 0.4 : graph <-> arrays, fresh-rebuild, scoring ---------------------
# Everything is kept in index space (node index 0..N-1 == row in coords) so a
# fresh scoring copy can be rebuilt cheaply (compute_metric mutates its input by
# matching, so each score needs a fresh graph). make_graph returns the node_ids
# so matches can be mapped back to indices regardless of id assignment.

_NODE_KEYSETS = [
    [K.NODE_ID, K.T, K.Z, K.Y, K.X],
    [K.NODE_ID, "t", "z", "y", "x"],
]

def _node_arrays(g):
    last = None
    for ks in _NODE_KEYSETS:
        try:
            df = g.node_attrs(attr_keys=ks)
            return (np.asarray(df[ks[0]].to_list()),
                    np.asarray(df[ks[1]].to_list(), dtype=np.float64),
                    np.asarray(df[ks[2]].to_list(), dtype=np.float64),
                    np.asarray(df[ks[3]].to_list(), dtype=np.float64),
                    np.asarray(df[ks[4]].to_list(), dtype=np.float64))
        except Exception as e:
            last = e
    raise last

def _edge_pairs(g):
    if g.num_edges() == 0:
        return []
    df = g.edge_attrs(attr_keys=[K.EDGE_SOURCE, K.EDGE_TARGET])
    return list(zip(df[K.EDGE_SOURCE].to_list(), df[K.EDGE_TARGET].to_list()))

def load_graph_arrays(g):
    """Graph -> (coords[N,4]=t,z,y,x, idx_edges list[(i,j)])."""
    nid, t, z, y, x = _node_arrays(g)
    id2i = {int(v): i for i, v in enumerate(nid)}
    coords = np.stack([t, z, y, x], axis=1)
    idx_edges = [(id2i[int(s)], id2i[int(tt)])
                 for s, tt in _edge_pairs(g)
                 if int(s) in id2i and int(tt) in id2i]
    return coords, idx_edges

def make_graph(coords, idx_edges):
    """Build a fresh tracksdata graph in index space; returns (graph, node_ids)."""
    g = td.graph.InMemoryGraph()
    for key in ("z", "y", "x"):
        g.add_node_attr_key(key, pl.Float64, -999999.0)
    node_ids = g.bulk_add_nodes([
        {"t": int(t), "z": float(z), "y": float(y), "x": float(x)}
        for t, z, y, x in coords
    ])
    if idx_edges:
        g.add_edge_attr_key("edge_prob", pl.Float64, 0.0)
        g.add_edge_attr_key("edge_dist", pl.Float64, 0.0)
        g.bulk_add_edges([
            {"source_id": node_ids[i], "target_id": node_ids[j],
             "edge_prob": 1.0,
             "edge_dist": float(np.linalg.norm((coords[i, 1:] - coords[j, 1:]) * np.asarray(VOXEL_SCALE)))}
            for (i, j) in idx_edges
        ])
    return g, node_ids

def score(coords, idx_edges, gt_graph, n_total, want_match=False):
    """Score an index-space graph vs GT. Returns metric dict; optionally the
    gt_node_id -> pred_index matching (from the scorer's own assignment)."""
    g, node_ids = make_graph(coords, idx_edges)
    id2idx = {int(nid): i for i, nid in enumerate(node_ids)}
    er = compute_metric(g, gt_graph, scale=VOXEL_SCALE, max_distance=MAX_DIST)
    d_tp, d_fp, d_fn = er.division_tp, er.division_fp, er.division_fn
    denom = d_tp + d_fp + d_fn
    div_j = (d_tp / denom) if denom > 0 else 0.0
    na = g.node_attrs(attr_keys=[K.NODE_ID, K.MATCHED_NODE_ID])
    pids = na[K.NODE_ID].to_list()
    mids = na[K.MATCHED_NODE_ID].to_list()
    matched_gt = [int(m) for m in mids if m is not None and int(m) != -1]
    recall = len(set(matched_gt)) / max(1, gt_graph.num_nodes())
    psm = per_sample_metrics(er, n_total, recall)
    adj = psm["adj_edge_jaccard"]
    out = {
        "adj": adj, "edge_j": psm["edge_jaccard"], "div_j": div_j,
        "score": (adj if adj == adj else 0.0) + DIV_WEIGHT * div_j,
        "e_tp": er.edge_tp, "e_fp": er.edge_fp, "e_fn": er.edge_fn,
        "d_tp": d_tp, "d_fp": d_fp, "d_fn": d_fn,
        "recall": recall, "n_pred": er.num_pred_nodes,
    }
    if want_match:
        gt2pred = {}
        for nid, m in zip(pids, mids):
            if m is not None and int(m) != -1:
                gt2pred[int(m)] = id2idx[int(nid)]
        out["gt2pred"] = gt2pred
    return out

def read_n_total(name):
    if GeffMetadata is None:
        return float("nan")
    try:
        meta = GeffMetadata.read(DATA_DIR / f"{name}.geff")
        v = (meta.extra or {}).get("estimated_number_of_nodes")
        return float(v) if v is not None else float("nan")
    except Exception:
        return float("nan")

print("helpers ready")

In [ ]:
# --- Phase 0.5 : load baselines + GT, extract GT divisions --------------------
def gt_adjacency(gt):
    children, parents = defaultdict(list), defaultdict(list)
    if gt.num_edges() > 0:
        df = gt.edge_attrs(attr_keys=[K.EDGE_SOURCE, K.EDGE_TARGET])
        for s, t in zip(df[K.EDGE_SOURCE].to_list(), df[K.EDGE_TARGET].to_list()):
            children[int(s)].append(int(t))
            parents[int(t)].append(int(s))
    return children, parents

videos = {}
for name in names:
    pg = td.graph.IndexedRXGraph.from_geff(str(Path(PRED_DIR) / f"{name}.geff"))
    if isinstance(pg, tuple):
        pg = pg[0]
    coords, base_edges = load_graph_arrays(pg)
    gt = open_dataset(DATA_DIR / name, require_tracks=True).tracks
    ch, pa = gt_adjacency(gt)
    gt_divs = [m for m in ch if len(ch[m]) >= 2]  # GT dividing nodes (out-degree >= 2)
    videos[name] = dict(coords=coords, base_edges=base_edges, gt=gt,
                        n_total=read_n_total(name), children=ch, parents=pa, gt_divs=gt_divs)
    print(f"{name}: pred_nodes={len(coords)} pred_edges={len(base_edges)} "
          f"gt_nodes={gt.num_nodes()} gt_divs={len(gt_divs)}")
print("total GT divisions:", sum(len(v['gt_divs']) for v in videos.values()))

In [ ]:
# --- Phase 0.6 : per-video baseline score + scorer matching ------------------
# One compute_metric per video gives (a) the baseline metric and (b) the exact
# gt_node -> pred_index matching the official scorer uses (so "detectable" is
# defined by the metric itself, not a home-grown NN gate).
for name, v in videos.items():
    base = score(v["coords"], v["base_edges"], v["gt"], v["n_total"], want_match=True)
    v["base"] = base
    v["gt2pred"] = base["gt2pred"]
    indeg = defaultdict(int)
    outadj = defaultdict(list)
    for (i, j) in v["base_edges"]:
        indeg[j] += 1
        outadj[i].append(j)
    v["indeg"], v["outadj"] = indeg, outadj

print("baseline (per video): adj / div_J / d_tp,fp,fn")
for name, v in videos.items():
    b = v["base"]
    print(f"  {name}: adj={b['adj']:.4f} div_J={b['div_j']:.4f} "
          f"(TP{b['d_tp']}/FP{b['d_fp']}/FN{b['d_fn']}) recall={b['recall']:.4f}")

In [ ]:
# --- Phase 0.7 : ORACLE family A (orphan-only) -------------------------------
# For each detectable GT division (mother + both daughters matched by the scorer):
#   if the mother's pred node already links to exactly one daughter, and the other
#   daughter's pred node is an orphan (in-degree 0), the oracle-correct edit is
#   add(mother_pred -> other_daughter_pred). We apply ALL such edits per video at
#   once (the true combined ceiling) and rescore.
def family_A_edits(v):
    g2p = v["gt2pred"]
    edits, detectable = [], 0
    for m in v["gt_divs"]:
        daughters = v["children"][m][:2]
        if m not in g2p or any(d not in g2p for d in daughters):
            continue
        detectable += 1
        Mi = g2p[m]
        Ds = [g2p[d] for d in daughters]
        linked = [d for d in Ds if d in v["outadj"].get(Mi, [])]
        if len(linked) != 1:
            continue                      # precondition (A): exactly one child present
        other = [d for d in Ds if d not in linked][0]
        if v["indeg"].get(other, 0) != 0:
            continue                      # not an orphan -> family B/C territory
        edits.append((Mi, other))
    return edits, detectable

rows = []
for name, v in videos.items():
    edits, detectable = family_A_edits(v)
    new_edges = v["base_edges"] + edits
    orc = score(v["coords"], new_edges, v["gt"], v["n_total"]) if edits else v["base"]
    b = v["base"]
    rows.append(dict(
        name=name, specimen=specimen_of(name),
        gt_divs=len(v["gt_divs"]), detectable=detectable, A_edits=len(edits),
        d_adj=orc["adj"] - b["adj"], d_divj=orc["div_j"] - b["div_j"],
        d_score=orc["score"] - b["score"],
        base_adj=b["adj"], orc_adj=orc["adj"],
        base_divj=b["div_j"], orc_divj=orc["div_j"],
    ))
    print(f"  {name}: A_edits={len(edits)}/{detectable}det/{len(v['gt_divs'])}gt "
          f"Dadj={rows[-1]['d_adj']:+.4f} DdivJ={rows[-1]['d_divj']:+.4f} "
          f"Dscore={rows[-1]['d_score']:+.4f}")
famA = pl.DataFrame(rows)
famA

In [ ]:
# --- Phase 0.8 : per-specimen aggregate + GATE 0 (family A) ------------------
def agg(df):
    return df.group_by("specimen").agg([
        pl.col("gt_divs").sum().alias("gt_divs"),
        pl.col("detectable").sum().alias("detectable"),
        pl.col("A_edits").sum().alias("recovered"),
        pl.col("d_adj").mean().alias("mean_d_adj"),
        pl.col("d_divj").mean().alias("mean_d_divj"),
        pl.col("d_score").mean().alias("mean_d_score"),
    ]).sort("specimen")

summary = agg(famA)
print(summary)
print()
detect_frac = famA["detectable"].sum() / max(1, famA["gt_divs"].sum())
print(f"daughter-detectability ceiling: {famA['detectable'].sum()}/{famA['gt_divs'].sum()} "
      f"= {detect_frac:.2%} of GT divisions have mother+both daughters detected")
print(f"family-A recoverable (orphan precondition): {famA['A_edits'].sum()} divisions")

both_pos = bool((summary["mean_d_score"] > 0).all())
print("\n=== GATE 0 (family A) ===")
print("PASS -> build Phase 1" if both_pos else
      "family A weak -> evaluate B/C (Phase 0.9) before deciding")
print("(discount local optimism ~0.5x before reading against the LB; require both specimens > 0.)")

## Phase 0.9 - families B / C (scaffold)

Family A alone is bounded by how often the true daughter is left an **orphan** by the linker.
Families **B** (steal a weakly-assigned child) and **C** (joint daughter selection) break that
ceiling but modify existing edges, so they move `adj` and carry the Version-7 edge-FP risk - the
oracle must read their per-specimen `Delta adj`, not only `div_J`.

The machinery is identical: enumerate an atomic edit set per mother, apply to `base_edges`
(remove + add), rescore with `score(...)`, and aggregate exactly as in 0.7/0.8. Fill the two
functions below after reading the family-A ceiling, then reuse Phase 0.8's `agg` / GATE logic.

In [ ]:
# --- Phase 0.9 : families B / C (to implement after A is read) ---------------
def family_B_edits(v):
    """Steal a weak child: allow D2 with an existing parent P where P->D2 is weak
    (long / low-confidence). Atomic edit = remove (P,D2) then add (M,D2)."""
    raise NotImplementedError("Phase 0.9 - implement once family A is read")

def family_C_edits(v):
    """Joint daughter selection: do not assume the current child of M is correct;
    enumerate {no-change, single-continuation, replace, divide, replace+divide}
    over M's next-frame top-k and keep the metric-best atomic set."""
    raise NotImplementedError("Phase 0.9 - implement once family A is read")

print("B/C scaffolded - enable after reading the family-A ceiling.")

## Optional (Phase-1 prep) - pre-ILP candidate-edge export

Not needed for the Phase-0 gate. When building Phase 1 features we need the transformer scores for
`M -> D2` even though the ILP did NOT select that edge (so it is absent from the saved graph).
Capture them by patching `predict_unet_transformer.predict_video` to persist, per consecutive frame
pair, the full `edge_logits_pair` matrix (raw logit + softmax) BEFORE the candidate/cap filter
(around lines 447-488), keyed by `(gi, gj)`. Save alongside `coords` so Phase 1 can attach
logit / rank / top-1-2 margin features to every candidate triplet.